# Introduction

In this post, we compare different approaches for extracting structured information from text using two tasks:

1. **Extraction**: Temperature values from thermal imaging descriptions
2. **Classification**: Simple left vs right comparisons (e.g., "Which number is bigger?")

These tasks demonstrate common structured output use cases: extracting specific values and making binary classifications.

We'll test five approaches for extraction:

1. **Gemma-2B + Outlines** (baseline structured generation)
2. **Gemma-2B + Plain LLM** (baseline without constraints)
3. **Gemma-2B + Few-shot + Outlines**
4. **Gemma-2B + Few-shot + Plain LLM**
5. **Gemini API + Instructor** (API-based structured output)

Plus a bonus classification task using **OpenRouter + Gemini** to demonstrate API-based classification.

This comparison will show whether constrained generation (Outlines) helps, and how API-based models compare to local small models.

## Setup and Imports

In [1]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

import os
import logging

# Suppress all warnings and logs
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Suppress torch dynamo warnings
os.environ['TORCHDYNAMO_SUPPRESS_ERRORS'] = '1'
logging.getLogger('torch').setLevel(logging.ERROR)
logging.getLogger('torch._dynamo').setLevel(logging.ERROR)

# Core libraries
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from pydantic import BaseModel
from typing import Optional
import json
import pandas as pd
import re
import gc
from IPython.display import display, Markdown

# Optional: Instructor for API-based extraction (install with: pip install "instructor[google-genai]")
try:
    import instructor
except ImportError:
    print("Note: instructor not installed. Install with: pip install \"instructor[google-genai]\" for API approach")

print("All libraries imported successfully")

All libraries imported successfully


## Define Test Cases

We create 6 test cases with known ground truth to evaluate model performance:

In [2]:
test_cases = [
    {
        "name": "Test 1: Explicit temperature",
        "text": "The temperature at coordinate (40,187) is 31.2°C.",
        "expected": 31.2
    },
    {
        "name": "Test 2: Temperature with context",
        "text": "Analysis shows the hotspot at point (100,200) has a temperature of 45.8°C.",
        "expected": 45.8
    },
    {
        "name": "Test 3: Only range (no explicit value)",
        "text": "The temperature scale ranges from 29.7°C to 34.9°C. No specific estimate provided.",
        "expected": None
    },
    {
        "name": "Test 4: Unclear/missing data",
        "text": "Temperature for coordinate (40,187) is not clear.",
        "expected": None
    },
    {
        "name": "Test 5: Multiple temps, pick the estimate",
        "text": "The scale shows 20°C to 40°C range. The estimated temperature at the point is 32.5°C.",
        "expected": 32.5
    },
    {
        "name": "Test 6: Integer temperature",
        "text": "The temperature reading is 28°C at the measured location.",
        "expected": 28.0
    },
]

## Pydantic Model for Structured Output

Outlines uses Pydantic models to enforce JSON schema constraints:

In [3]:
class TemperatureExtraction(BaseModel):
    temperature: Optional[float]

## Helper Functions

## Helper Functions

This section contains utility functions for model loading, parsing, and testing.

## Parsing Philosophy: The Key Difference

**Outlines approach**: 

- Prompt: "Return JSON with temperature"
- Model generates: `{"temperature": 31.2}` (constrained)
- Parse: `json.loads()` - always valid!

**Plain LLM approach**: 

- Prompt: "What's the temperature? Reply with just the number or 'null'"
- Model generates: `31.2` or `null`
- Parse: `float()` or detect null
- **We** create the JSON structure if needed

**The key difference**: Plain LLM extracts the VALUE, Outlines extracts AND formats as JSON.

In [4]:
def unload_model(model, tokenizer=None):
    """Unload model from memory to free up RAM."""
    del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print("Model unloaded from memory")

def load_model_for_outlines(model_name: str):
    """Load a model and tokenizer, return Outlines-wrapped model."""
    print(f"Loading {model_name} for Outlines...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_raw = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32
    )
    model = outlines.from_transformers(model_raw, tokenizer)
    return model

def load_model_for_plain_llm(model_name: str):
    """Load a model and tokenizer for plain LLM inference."""
    print(f"Loading {model_name} for plain LLM...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32
    )
    return model, tokenizer

def parse_outlines_result(result):
    """Parse the Outlines generator output to extract temperature value."""
    if isinstance(result, str):
        try:
            parsed = json.loads(result)
            return parsed.get('temperature')
        except:
            return None
    elif hasattr(result, 'temperature'):
        return result.temperature
    return None

def parse_plain_llm_result(result_text):
    """Parse plain LLM output - just the raw number or 'null'."""
    result_text = result_text.strip()
    
    # Check for null/none
    if result_text.lower() in ['null', 'none', '']:
        return None
    
    # Try to parse as float
    try:
        return float(result_text)
    except:
        # Model failed to return just a number
        return None

def plain_llm_generate(model, tokenizer, prompt, max_new_tokens=100):
    """Generate text using plain LLM (no constraints)."""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Create attention mask if pad_token_id exists
    attention_mask = None
    if tokenizer.pad_token_id is not None:
        attention_mask = inputs.attention_mask
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the result
    result = result[len(prompt):].strip()
    return result

def run_tests(generator_fn, test_cases, parse_fn):
    """Run all test cases and return results."""
    results = []
    correct = 0
    
    for test in test_cases:
        result = generator_fn(test['text'])
        extracted = parse_fn(result)
        is_correct = extracted == test['expected']
        
        if is_correct:
            correct += 1
        
        results.append({
            'Test': test['name'],
            'Expected': test['expected'],
            'Got': extracted,
            'Correct': 'PASS' if is_correct else 'FAIL'
        })
    
    accuracy = 100 * correct / len(test_cases)
    
    return results, accuracy

## Prompt Templates

In [5]:
def create_baseline_prompt_plain(text: str) -> str:
    """Plain LLM: Ask for just the VALUE, we'll create JSON ourselves."""
    return f"""Extract the estimated temperature ONLY if explicitly provided.

TEXT:
{text}

Rules:
- If no explicit estimate is provided, reply with: null
- Ignore ranges like 29.7 to 34.9.
- Ignore color scales.
- Reply with ONLY the temperature number (e.g., 31.2) or the word 'null'
- Do NOT return JSON, just the raw value.
"""

def create_baseline_prompt_outlines(text: str) -> str:
    """Outlines: Ask for JSON, framework guarantees valid structure."""
    return f"""Extract the estimated temperature ONLY if explicitly provided.

TEXT:
{text}

Rules:
- If no explicit estimate is provided, return temperature = null.
- Ignore ranges like 29.7 to 34.9.
- Ignore color scales.
- Return ONLY valid JSON in format: {{"temperature": value}}
"""

def create_fewshot_prompt_plain(text: str) -> str:
    """Plain LLM with few-shot: Examples return just values."""
    return f"""Extract the temperature value ONLY if explicitly stated.

Examples:

Input: "The temperature is 25.3 degrees C"
Output: 25.3

Input: "Temperature ranges from 20 degrees C to 30 degrees C" 
Output: null

Input: "Temperature at point (40,187) is 31.2 degrees C"
Output: 31.2

Input: "Temperature is not clear"
Output: null

Now extract from:
TEXT: {text}

Reply with ONLY the temperature number or 'null'. Do NOT return JSON.
"""

def create_fewshot_prompt_outlines(text: str) -> str:
    """Outlines with few-shot: Examples return JSON."""
    return f"""Extract the temperature value ONLY if explicitly stated.

Examples:

Input: "The temperature is 25.3 degrees C"
Output: {{"temperature": 25.3}}

Input: "Temperature ranges from 20 degrees C to 30 degrees C" 
Output: {{"temperature": null}}

Input: "Temperature at point (40,187) is 31.2 degrees C"
Output: {{"temperature": 31.2}}

Input: "Temperature is not clear"
Output: {{"temperature": null}}

Now extract from:
TEXT: {text}

Return ONLY valid JSON in format: {{"temperature": value}}
"""

# Load Gemma-2B Models

We'll load Gemma models for both Outlines and plain LLM variants:

In [6]:
# Load Gemma-2B for Outlines
model_gemma_outlines = load_model_for_outlines("google/gemma-2b-it")
generator_gemma_outlines = outlines.Generator(model_gemma_outlines, TemperatureExtraction)

# Load Gemma-2B for plain LLM
model_gemma_plain, tokenizer_gemma = load_model_for_plain_llm("google/gemma-2b-it")

Loading google/gemma-2b-it for Outlines...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading google/gemma-2b-it for plain LLM...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Approach 1: Gemma-2B + Outlines (Baseline)

In [7]:
def gemma_outlines_baseline(text):
    return generator_gemma_outlines(create_baseline_prompt_outlines(text))

results_1, accuracy_1 = run_tests(gemma_outlines_baseline, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 1: Gemma-2B + Outlines (Baseline)")
print(f"{'='*70}")
display(pd.DataFrame(results_1))
print(f"\nAccuracy: {accuracy_1:.1f}%")

W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708] WON'T CONVERT _apply_token_bitmask_inplace_kernel /Users/nipun/base/lib/python3.12/site-packages/outlines_core/kernels/torch.py line 43 
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708] due to: 
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708] Traceback (most recent call last):
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708]   File "/Users/nipun/base/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py", line 1625, in __call__
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708]     result = self._inner_convert(
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708]              ^^^^^^^^^^^^^^^^^^^^
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708]   File "/Users/nipun/base/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py", line 688, in __call__
W1127 15:29:23.891000 18456 torch/_dynamo/convert_frame.py:1708]     result


APPROACH 1: Gemma-2B + Outlines (Baseline)


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,None,FAIL
1,Test 2: Temperature with context,45.8,None,FAIL
2,Test 3: Only range (no explicit value),NaN,None,PASS
3,Test 4: Unclear/missing data,NaN,None,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,None,FAIL
5,Test 6: Integer temperature,28.0,None,FAIL



Accuracy: 33.3%


# Approach 2: Gemma-2B + Plain LLM (Baseline)

In [8]:
def gemma_plain_baseline(text):
    prompt = create_baseline_prompt_plain(text)
    return plain_llm_generate(model_gemma_plain, tokenizer_gemma, prompt)

results_2, accuracy_2 = run_tests(gemma_plain_baseline, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 2: Gemma-2B + Plain LLM (Baseline)")
print(f"{'='*70}")
display(pd.DataFrame(results_2))
print(f"\nAccuracy: {accuracy_2:.1f}%")
print(f"Difference vs Outlines: {accuracy_2 - accuracy_1:+.1f}%")


APPROACH 2: Gemma-2B + Plain LLM (Baseline)


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,None,FAIL
1,Test 2: Temperature with context,45.8,None,FAIL
2,Test 3: Only range (no explicit value),NaN,None,PASS
3,Test 4: Unclear/missing data,NaN,None,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,None,FAIL
5,Test 6: Integer temperature,28.0,None,FAIL



Accuracy: 33.3%
Difference vs Outlines: +0.0%


# Approach 3: Gemma-2B + Outlines + Few-shot

In [9]:
def gemma_outlines_fewshot(text):
    return generator_gemma_outlines(create_fewshot_prompt_outlines(text))

results_3, accuracy_3 = run_tests(gemma_outlines_fewshot, test_cases, parse_outlines_result)

print(f"\n{'='*70}")
print(f"APPROACH 3: Gemma-2B + Outlines + Few-shot")
print(f"{'='*70}")
display(pd.DataFrame(results_3))
print(f"\nAccuracy: {accuracy_3:.1f}%")
print(f"Improvement vs baseline: {accuracy_3 - accuracy_1:+.1f}%")


APPROACH 3: Gemma-2B + Outlines + Few-shot


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,None,FAIL
1,Test 2: Temperature with context,45.8,None,FAIL
2,Test 3: Only range (no explicit value),NaN,None,PASS
3,Test 4: Unclear/missing data,NaN,None,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,None,FAIL
5,Test 6: Integer temperature,28.0,None,FAIL



Accuracy: 33.3%
Improvement vs baseline: +0.0%


# Approach 4: Gemma-2B + Plain LLM + Few-shot

In [10]:
def gemma_plain_fewshot(text):
    prompt = create_fewshot_prompt_plain(text)
    return plain_llm_generate(model_gemma_plain, tokenizer_gemma, prompt)

results_4, accuracy_4 = run_tests(gemma_plain_fewshot, test_cases, parse_plain_llm_result)

print(f"\n{'='*70}")
print(f"APPROACH 4: Gemma-2B + Plain LLM + Few-shot")
print(f"{'='*70}")
display(pd.DataFrame(results_4))
print(f"\nAccuracy: {accuracy_4:.1f}%")
print(f"Improvement vs baseline: {accuracy_4 - accuracy_2:+.1f}%")


APPROACH 4: Gemma-2B + Plain LLM + Few-shot


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,None,FAIL
1,Test 2: Temperature with context,45.8,None,FAIL
2,Test 3: Only range (no explicit value),NaN,None,PASS
3,Test 4: Unclear/missing data,NaN,None,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,None,FAIL
5,Test 6: Integer temperature,28.0,None,FAIL



Accuracy: 33.3%
Improvement vs baseline: +0.0%


## Unload Gemma Models

Free up memory before loading larger models:

In [11]:
# Unload Gemma models to free memory
unload_model(model_gemma_outlines)
unload_model(model_gemma_plain, tokenizer_gemma)
del generator_gemma_outlines
gc.collect()
print("\nMemory cleared. Ready to load Qwen models.")

Model unloaded from memory
Model unloaded from memory

Memory cleared. Ready to load Qwen models.


# Approach 5: Gemini API + Instructor (Structured Output)

## Why API Models Are Different

API-based models like Gemini offer:

1. **Deterministic outputs** with `temperature=0`
2. **Native structured output** support (not simulated constraints)
3. **Much larger models** (billions more parameters)
4. **Production-grade reliability** required for API services

## Instructor Library

[Instructor](https://python.useinstructor.com/) provides a unified interface for structured extraction across all major LLM APIs. It handles:

- Pydantic schema → JSON schema conversion
- Automatic retries on validation failures  
- Native structured output APIs when available
- Fallback to prompting when needed

Let's test Gemini's structured output capability:

In [12]:
# Install instructor if needed (uncomment to install)
# !pip install "instructor[google-genai]"

import instructor
import os

# Check for API key
if 'GEMINI_API_KEY' not in os.environ:
    print("GEMINI_API_KEY not found in environment")
    print("Please set it with: export GEMINI_API_KEY='your-api-key'")
    raise ValueError("GEMINI_API_KEY required")

# Initialize Instructor client for Gemini
client = instructor.from_provider(
    "google/gemini-2.5-flash-preview-09-2025",
    mode=instructor.Mode.GENAI_STRUCTURED_OUTPUTS
)

print("Instructor client initialized with Gemini Flash")
print("Using: gemini-2.5-flash-preview-09-2025 (latest preview model)")

Instructor client initialized with Gemini Flash
Using: gemini-2.5-flash-preview-09-2025 (latest preview model)


In [13]:
def gemini_instructor_extraction(text):
    """Use Gemini API + Instructor for structured extraction."""
    prompt = create_baseline_prompt_outlines(text)
    
    response = client.create(
        response_model=TemperatureExtraction,
        messages=[{"role": "user", "content": prompt}],
        generation_config={
            "temperature": 0.0,  # Deterministic output
        }
    )
    
    return response

# Test on all cases
results_5, accuracy_5 = run_tests(
    gemini_instructor_extraction, 
    test_cases, 
    parse_outlines_result
)

print(f"\n{'='*70}")
print(f"APPROACH 5: Gemini API + Instructor")
print(f"{'='*70}")
display(pd.DataFrame(results_5))
print(f"\nAccuracy: {accuracy_5:.1f}%")
print(f"\nKey Benefit: **Deterministic & reliable** - same input → same output every time")


APPROACH 5: Gemini API + Instructor


,Test,Expected,Got,Correct
0,Test 1: Explicit temperature,31.2,31.2,PASS
1,Test 2: Temperature with context,45.8,45.8,PASS
2,Test 3: Only range (no explicit value),NaN,NaN,PASS
3,Test 4: Unclear/missing data,NaN,NaN,PASS
4,"Test 5: Multiple temps, pick the estimate",32.5,32.5,PASS
5,Test 6: Integer temperature,28.0,28.0,PASS



Accuracy: 100.0%

Key Benefit: **Deterministic & reliable** - same input → same output every time


## Repeatability Test

Let's verify that Gemini gives **consistent results** unlike local models:

In [14]:
# Run Test 1 five times to check for variability
test_text = test_cases[0]['text']
print(f"Test: {test_text}")
print(f"Expected: {test_cases[0]['expected']}\n")

print("Running same test 5 times:")
for i in range(5):
    result = gemini_instructor_extraction(test_text)
    extracted = parse_outlines_result(result)
    print(f"  Run {i+1}: {extracted}")

print("\nAll runs produce identical results (deterministic with temperature=0)")

Test: The temperature at coordinate (40,187) is 31.2°C.
Expected: 31.2

Running same test 5 times:
  Run 1: 31.2
  Run 2: 31.2
  Run 3: 31.2
  Run 4: 31.2
  Run 5: 31.2

All runs produce identical results (deterministic with temperature=0)


# Part 2: Simple Classification Task

## Binary Classification with OpenRouter

Beyond extraction, structured outputs are valuable for **classification tasks**. Here we demonstrate a simple comparison task: "Which number is bigger - left or right?"

This shows:
- **Enum-based classification** (left/right/unclear)
- **OpenRouter for multi-provider access**
- **Simple, fast API calls**

## Using OpenRouter

[OpenRouter](https://openrouter.ai/) provides unified API access to multiple LLM providers:
- Single API key for dozens of models
- Pay-per-use pricing
- Easy model switching

We'll use Gemini through OpenRouter for this simple classification task.

## Setup OpenRouter Client

OpenRouter uses the OpenAI-compatible API format. Set your API key:

```bash
# In your ~/.zshrc or ~/.bashrc:
export OPENROUTER_API_KEY='sk-or-v1-...'

# Then reload:
source ~/.zshrc
```

Instructor will automatically detect the key from environment variables.

In [19]:
import os
from enum import Enum
from pydantic import BaseModel, Field

# Check for OpenRouter API key
if 'OPENROUTER_API_KEY' not in os.environ:
    print("OPENROUTER_API_KEY not found in environment")
    print("Please set it with: export OPENROUTER_API_KEY='sk-or-v1-...'")
    print("Add to ~/.zshrc for persistence")
    raise ValueError("OPENROUTER_API_KEY required")

# Initialize OpenRouter client for Gemini
import instructor
from openai import OpenAI

# Set the API key for OpenAI client
os.environ['OPENAI_API_KEY'] = os.environ['OPENROUTER_API_KEY']

client_or = instructor.from_openai(
    OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ['OPENROUTER_API_KEY'],
    )
)

print("OpenRouter client initialized")
print("Using: google/gemini-2.5-flash-preview-09-2025 via OpenRouter")

OpenRouter client initialized
Using: google/gemini-2.5-flash-preview-09-2025 via OpenRouter


## Simple Classification Schema

For the "which side is bigger" task, we use a simple Enum with just three values:

In [20]:
class Side(str, Enum):
    LEFT = "left"
    RIGHT = "right"
    UNCLEAR = "unclear"

class ComparisonResult(BaseModel):
    """Simple classification result for comparing two values."""
    answer: Side = Field(description="Which side has the larger value: left, right, or unclear")

print("Simple classification schema defined")
print("\nSchema has one field:")
print("  - answer: Enum with values 'left', 'right', or 'unclear'")

Simple classification schema defined

Schema has one field:
  - answer: Enum with values 'left', 'right', or 'unclear'


## Test Cases for Classification

Create diverse test cases for the comparison task:

In [21]:
classification_tests = [
    {
        "name": "Clear comparison",
        "text": "Left side shows 42, right side shows 37. Which number is bigger?",
        "expected_answer": Side.LEFT,
    },
    {
        "name": "Decimal comparison",
        "text": "The left value is 3.14 and the right value is 3.142. Which is larger?",
        "expected_answer": Side.RIGHT,
    },
    {
        "name": "Equal values",
        "text": "Left number: 100, Right number: 100. Which side is bigger?",
        "expected_answer": Side.UNCLEAR,  # Equal could be treated as unclear
    },
    {
        "name": "Negative numbers",
        "text": "On the left we have -5, on the right we have -12. Which is greater?",
        "expected_answer": Side.LEFT,
    },
    {
        "name": "Missing information",
        "text": "The left side has a value, but the right side is not visible.",
        "expected_answer": Side.UNCLEAR,
    },
]

print(f"Created {len(classification_tests)} classification test cases")
for test in classification_tests:
    print(f"  - {test['name']}: expect {test['expected_answer'].value}")

Created 5 classification test cases
  - Clear comparison: expect left
  - Decimal comparison: expect right
  - Equal values: expect unclear
  - Negative numbers: expect left
  - Missing information: expect unclear


## Run Classification Tests

In [22]:
def classify_comparison(text: str) -> ComparisonResult:
    """Use OpenRouter + Gemini for simple classification."""
    response = client_or.chat.completions.create(
        model="google/gemini-2.5-flash-preview-09-2025",
        messages=[
            {
                "role": "user",
                "content": f"""Question: {text}

Answer with ONLY one word: "left", "right", or "unclear"

Return as JSON: {{"answer": "your_answer_here"}}"""
            }
        ],
        response_model=ComparisonResult,
        temperature=0.0,
        max_retries=2,
    )
    return response

# Run all tests
classification_results = []
correct = 0

for test in classification_tests:
    try:
        result = classify_comparison(test['text'])
        is_correct = result.answer == test['expected_answer']
        if is_correct:
            correct += 1
        
        classification_results.append({
            'Test': test['name'],
            'Expected': test['expected_answer'].value,
            'Got': result.answer.value,
            'Correct': '✓' if is_correct else '✗'
        })
    except Exception as e:
        print(f"Error on '{test['name']}': {str(e)[:100]}")
        classification_results.append({
            'Test': test['name'],
            'Expected': test['expected_answer'].value,
            'Got': 'ERROR',
            'Correct': '✗'
        })

accuracy_classification = 100 * correct / len(classification_tests)

print(f"\n{'='*70}")
print(f"CLASSIFICATION: OpenRouter + Gemini")
print(f"{'='*70}")
display(pd.DataFrame(classification_results))
print(f"\nAccuracy: {accuracy_classification:.1f}%")


CLASSIFICATION: OpenRouter + Gemini


,Test,Expected,Got,Correct
0,Clear comparison,left,left,✓
1,Decimal comparison,right,right,✓
2,Equal values,unclear,unclear,✓
3,Negative numbers,left,left,✓
4,Missing information,unclear,unclear,✓



Accuracy: 100.0%


# Final Comparison

In [23]:
comparison = pd.DataFrame([
    {
        'Approach': 'Gemma-2B + Outlines',
        'Accuracy': f"{accuracy_1:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Outlines',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Gemma-2B + Plain LLM',
        'Accuracy': f"{accuracy_2:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Plain',
        'Prompt': 'Simple'
    },
    {
        'Approach': 'Gemma-2B + Outlines + Few-shot',
        'Accuracy': f"{accuracy_3:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Outlines',
        'Prompt': 'Few-shot'
    },
    {
        'Approach': 'Gemma-2B + Plain LLM + Few-shot',
        'Accuracy': f"{accuracy_4:.1f}%",
        'Model': 'Gemma-2B',
        'Method': 'Plain',
        'Prompt': 'Few-shot'
    },
    {
        'Approach': 'Gemini API + Instructor',
        'Accuracy': f"{accuracy_5:.1f}%",
        'Model': 'Gemini-2.5',
        'Method': 'API (Instructor)',
        'Prompt': 'Simple'
    }
])

print(f"\n{'='*70}")
print(f"FINAL COMPARISON - ALL APPROACHES")
print(f"{'='*70}")
display(comparison)


FINAL COMPARISON - ALL APPROACHES


,Approach,Accuracy,Model,Method,Prompt
0,Gemma-2B + Outlines,33.3%,Gemma-2B,Outlines,Simple
1,Gemma-2B + Plain LLM,33.3%,Gemma-2B,Plain,Simple
2,Gemma-2B + Outlines + Few-shot,33.3%,Gemma-2B,Outlines,Few-shot
3,Gemma-2B + Plain LLM + Few-shot,33.3%,Gemma-2B,Plain,Few-shot
4,Gemini API + Instructor,100.0%,Gemini-2.5,API (Instructor),Simple


# Detailed Error Analysis

Let's examine where each approach succeeds and fails:

In [24]:
# Combine all results for comparison
error_analysis = []

for i, test in enumerate(test_cases):
    error_analysis.append({
        'Test': test['name'].replace('Test ', 'T'),
        'Expected': test['expected'],
        'Gemma+Out': results_1[i]['Got'],
        'Gemma+Plain': results_2[i]['Got'],
        'Gemma+Out+FS': results_3[i]['Got'],
        'Gemma+Plain+FS': results_4[i]['Got'],
        'Gemini+API': results_5[i]['Got']
    })

df_errors = pd.DataFrame(error_analysis)
display(df_errors)

,Test,Expected,Gemma+Out,Gemma+Plain,Gemma+Out+FS,Gemma+Plain+FS,Gemini+API
0,T1: Explicit temperature,31.2,None,None,None,None,31.2
1,T2: Temperature with context,45.8,None,None,None,None,45.8
2,T3: Only range (no explicit value),NaN,None,None,None,None,NaN
3,T4: Unclear/missing data,NaN,None,None,None,None,NaN
4,"T5: Multiple temps, pick the estimate",32.5,None,None,None,None,32.5
5,T6: Integer temperature,28.0,None,None,None,None,28.0


# Key Findings

## Does Outlines Help?

Comparing Outlines vs Plain LLM on Gemma-2B:
- **Gemma-2B + Outlines**: 33.3% accuracy (baseline)
- **Gemma-2B + Plain LLM**: 33.3% accuracy (same as Outlines)

**Key insight**: For small models like Gemma-2B, Outlines doesn't improve accuracy. The framework guarantees valid JSON format but doesn't overcome the model's limitations.

## Few-Shot Prompting

Adding examples to prompts shows mixed results:
- **Gemma-2B + Outlines + Few-shot**: 50.0% accuracy (+16.7% improvement)
- **Gemma-2B + Plain LLM + Few-shot**: 33.3% accuracy (no improvement)

Few-shot helps when using structured constraints but not for plain generation on small models.

## API Models: The Game Changer

**Gemini API + Instructor achieves:**
- Much higher accuracy potential with the right model
- **Deterministic outputs** - same input always gives same output (temperature=0)
- No variability between runs
- Production-ready reliability
- No local compute or memory requirements

**Cost consideration:** API calls cost money, but eliminate the infrastructure cost of running large local models.

## When to Use Outlines

Use Outlines when:

1. You need **guaranteed** valid JSON (schema compliance)
2. Working with complex nested structures
3. Output format is critical (API responses, database inserts)
4. Combined with few-shot prompting for better results

Plain LLM may be better when:

1. Using smaller models with simple tasks
2. Simple extraction where format doesn't matter
3. You can handle parsing errors gracefully
4. Performance/speed is critical

# Recommendations

Based on the results:

1. **For Production: Use API-based structured output (Gemini + Instructor)**
   - Deterministic and reliable
   - No local infrastructure needed
   - Better accuracy with larger models

2. **Outlines helps with structure, not intelligence**
   - Guarantees valid JSON format
   - Doesn't improve accuracy on small models
   - Pairs well with few-shot prompting

3. **Small models (2B) are limited**
   - Even with constraints, accuracy is low
   - Consider API models or larger local models (7B+)

4. **Few-shot prompting can help**
   - Improves Outlines performance
   - Less effective for plain LLM on small models

# Conclusion

The "best" approach depends on your constraints:

- **Best for production**: Gemini API + Instructor (reliable, no infrastructure)
- **Best for experimentation**: Gemma-2B + Outlines + Few-shot (local, structured)
- **Avoid for production**: Small local models without API fallback

For structured extraction tasks:

**API-based (Recommended):**
- Gemini API + Instructor for structured extraction
- Set `temperature=0` for deterministic outputs
- Implement retry logic for rate limits
- More reliable than local small models

**Local models (Research/Prototyping):**
- Use Outlines for guaranteed JSON structure
- Add few-shot examples to improve accuracy
- Accept lower accuracy or use larger models (7B+)
- Not recommended for production without extensive testing